# Weather Pipeline: Raw API to Reduced Dataset

Build one notebook flow that starts with Open-Meteo weather data and ends with a reduced feature dataset for household load prediction.

## Notebook Roadmap

This notebook is organized into seven simple steps:

1. Load paths, location settings, and shared weather variables.
2. Load the household timestamp calendar and the first target load.
3. Fetch raw historical weather data.
4. Convert raw API output into one clean 15-minute UTC weather table.
5. Create model-ready weather features.
6. Reduce the feature set for `residential1`.
7. Reuse the same reduced structure for future API calls and save outputs.

The notebook is meant as a clear workflow first. Later, the same logic can move into reusable Python modules.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config.json"
HOUSEHOLD_DATA_PATH = PROJECT_ROOT / "data" / "household_data_15min_singleindex.csv"
RAW_WEATHER_OUTPUT_PATH = PROJECT_ROOT / "data" / "weather_raw_hourly.csv"
FULL_WEATHER_OUTPUT_PATH = PROJECT_ROOT / "data" / "weather_full_15min.csv"
REDUCED_WEATHER_OUTPUT_PATH = PROJECT_ROOT / "data" / "weather_reduced_residential1.csv"
REDUCTION_SPEC_PATH = PROJECT_ROOT / "models" / "weather_feature_spec_residential1.pkl"

with CONFIG_PATH.open("r", encoding="utf-8") as file_handle:
    config = json.load(file_handle)

LATITUDE = config["lat"]
LONGITUDE = config["lon"]

LATITUDE, LONGITUDE

(47.659216, 9.1750718)

## Step 1: Define Shared Weather Variables

Keep one shared weather schema that can be used for historical data and future API calls.

In [3]:
COMMON_HOURLY_WEATHER_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "global_tilted_irradiance",
    "sunshine_duration",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
]

DERIVED_WEATHER_FEATURES = [
    "heating_degree_18c",
    "cooling_degree_22c",
    "is_raining",
    "is_snowing",
    "wind_u_10m",
    "wind_v_10m",
    "is_dark",
]

COMMON_HOURLY_WEATHER_VARS

['temperature_2m',
 'relative_humidity_2m',
 'dew_point_2m',
 'apparent_temperature',
 'precipitation',
 'rain',
 'snowfall',
 'snow_depth',
 'weather_code',
 'pressure_msl',
 'surface_pressure',
 'cloud_cover',
 'cloud_cover_low',
 'cloud_cover_mid',
 'cloud_cover_high',
 'shortwave_radiation',
 'direct_radiation',
 'diffuse_radiation',
 'global_tilted_irradiance',
 'sunshine_duration',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'et0_fao_evapotranspiration',
 'vapour_pressure_deficit']

## Step 2: Load Household Calendar And Target

Use the household timestamps as the calendar we want to match. Start with `residential1` as the first target for feature reduction.

In [4]:
household_df = pd.read_csv(HOUSEHOLD_DATA_PATH, parse_dates=["utc_timestamp"])
household_df["utc_timestamp"] = pd.to_datetime(household_df["utc_timestamp"], utc=True)
household_df = household_df.sort_values("utc_timestamp").reset_index(drop=True)

household_calendar = household_df[["utc_timestamp"]].drop_duplicates().reset_index(drop=True)

raw_target_cols = [
    "DE_KN_residential1_grid_import",
    "DE_KN_residential1_grid_export",
    "DE_KN_residential1_pv",
]
available_target_cols = [column for column in raw_target_cols if column in household_df.columns]
household_df[available_target_cols] = household_df[available_target_cols].diff()

household_df["residential1_load"] = 0.0
if "DE_KN_residential1_grid_import" in household_df.columns:
    household_df["residential1_load"] += household_df["DE_KN_residential1_grid_import"].fillna(0.0)
if "DE_KN_residential1_grid_export" in household_df.columns:
    household_df["residential1_load"] -= household_df["DE_KN_residential1_grid_export"].fillna(0.0)
if "DE_KN_residential1_pv" in household_df.columns:
    household_df["residential1_load"] += household_df["DE_KN_residential1_pv"].fillna(0.0)

household_calendar.head()

,utc_timestamp
0,2014-12-11 17:45:00+00:00
1,2014-12-11 18:00:00+00:00
2,2014-12-11 18:15:00+00:00
3,2014-12-11 18:30:00+00:00
4,2014-12-11 18:45:00+00:00


## Step 3: Fetch Raw Historical Weather Data

Fetch raw hourly data first. Later this should call one shared weather function for both historical backfill and future forecast runs.

In [5]:
import requests

ARCHIVE_ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"
REQUEST_TIMEOUT_SECONDS = 60

historical_start_date = household_calendar["utc_timestamp"].min().date()
historical_end_date = household_calendar["utc_timestamp"].max().date()

month_periods = pd.period_range(start=historical_start_date, end=historical_end_date, freq="M")
chunk_frames = []

for period in month_periods:
    chunk_start = max(historical_start_date, period.start_time.date())
    chunk_end = min(historical_end_date, period.end_time.date())

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": chunk_start.isoformat(),
        "end_date": chunk_end.isoformat(),
        "hourly": ",".join(COMMON_HOURLY_WEATHER_VARS),
        "timezone": "UTC",
    }

    if "global_tilted_irradiance" in COMMON_HOURLY_WEATHER_VARS and {"tilt", "azimuth"}.issubset(config):
        params["tilt"] = config["tilt"]
        params["azimuth"] = config["azimuth"]

    response = requests.get(ARCHIVE_ENDPOINT, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    payload = response.json()

    if isinstance(payload, dict) and payload.get("error"):
        reason = payload.get("reason", "Unknown API error")
        raise ValueError(f"Open-Meteo archive API error for {chunk_start} -> {chunk_end}: {reason}")

    hourly_payload = payload.get("hourly", {})
    if "time" not in hourly_payload:
        raise ValueError(
            f"Missing 'time' in hourly payload for {chunk_start} -> {chunk_end}. "
            f"Available keys: {list(hourly_payload.keys())}"
        )

    chunk_df = pd.DataFrame(hourly_payload)
    chunk_df["time"] = pd.to_datetime(chunk_df["time"], utc=True, errors="coerce")
    chunk_df = chunk_df.dropna(subset=["time"]).sort_values("time")

    for weather_col in COMMON_HOURLY_WEATHER_VARS:
        if weather_col not in chunk_df.columns:
            chunk_df[weather_col] = np.nan

    chunk_df = chunk_df[["time", *COMMON_HOURLY_WEATHER_VARS]].copy()
    chunk_df["source"] = "open_meteo_archive"

    chunk_frames.append(chunk_df)
    print(f"Fetched {chunk_start} -> {chunk_end}: {len(chunk_df)} rows")

if not chunk_frames:
    raise ValueError("No historical weather chunks were fetched.")

raw_weather_df = pd.concat(chunk_frames, ignore_index=True)
raw_weather_df = raw_weather_df.sort_values("time").drop_duplicates(subset=["time"], keep="first")
raw_weather_df = raw_weather_df.reset_index(drop=True)

RAW_WEATHER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
raw_weather_df.to_csv(RAW_WEATHER_OUTPUT_PATH, index=False)

print(
    f"Historical range: {historical_start_date} -> {historical_end_date} | "
    f"rows={len(raw_weather_df)} | cols={raw_weather_df.shape[1]}"
)
print(f"Saved raw hourly weather data to: {RAW_WEATHER_OUTPUT_PATH}")
raw_weather_df.head()

Fetched 2014-12-11 -> 2014-12-31: 504 rows
Fetched 2015-01-01 -> 2015-01-31: 744 rows
Fetched 2015-02-01 -> 2015-02-28: 672 rows
Fetched 2015-03-01 -> 2015-03-31: 744 rows
Fetched 2015-04-01 -> 2015-04-30: 720 rows
Fetched 2015-05-01 -> 2015-05-31: 744 rows
Fetched 2015-06-01 -> 2015-06-30: 720 rows
Fetched 2015-07-01 -> 2015-07-31: 744 rows
Fetched 2015-08-01 -> 2015-08-31: 744 rows
Fetched 2015-09-01 -> 2015-09-30: 720 rows
Fetched 2015-10-01 -> 2015-10-31: 744 rows
Fetched 2015-11-01 -> 2015-11-30: 720 rows
Fetched 2015-12-01 -> 2015-12-31: 744 rows
Fetched 2016-01-01 -> 2016-01-31: 744 rows
Fetched 2016-02-01 -> 2016-02-29: 696 rows
Fetched 2016-03-01 -> 2016-03-31: 744 rows
Fetched 2016-04-01 -> 2016-04-30: 720 rows
Fetched 2016-05-01 -> 2016-05-31: 744 rows
Fetched 2016-06-01 -> 2016-06-30: 720 rows
Fetched 2016-07-01 -> 2016-07-31: 744 rows
Fetched 2016-08-01 -> 2016-08-31: 744 rows
Fetched 2016-09-01 -> 2016-09-30: 720 rows
Fetched 2016-10-01 -> 2016-10-31: 744 rows
Fetched 201

,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,...,direct_radiation,diffuse_radiation,global_tilted_irradiance,sunshine_duration,wind_speed_10m,wind_direction_10m,wind_gusts_10m,et0_fao_evapotranspiration,vapour_pressure_deficit,source
0,2014-12-11 00:00:00+00:00,4.2,88,2.4,1.1,0.1,0.1,0.0,0.0,51,...,0.0,0.0,0.0,0.0,9.7,211,18.4,0.0,0.10,open_meteo_archive
1,2014-12-11 01:00:00+00:00,4.5,88,2.8,1.4,0.2,0.2,0.0,0.0,51,...,0.0,0.0,0.0,0.0,10.5,232,22.0,0.0,0.10,open_meteo_archive
2,2014-12-11 02:00:00+00:00,4.6,90,3.1,1.1,0.3,0.3,0.0,0.0,51,...,0.0,0.0,0.0,0.0,12.9,240,26.3,0.0,0.09,open_meteo_archive
3,2014-12-11 03:00:00+00:00,4.6,91,3.3,1.0,0.3,0.3,0.0,0.0,51,...,0.0,0.0,0.0,0.0,14.2,240,29.5,0.0,0.07,open_meteo_archive
4,2014-12-11 04:00:00+00:00,4.4,93,3.3,0.8,0.3,0.3,0.0,0.0,51,...,0.0,0.0,0.0,0.0,13.9,239,30.2,0.0,0.06,open_meteo_archive


## Step 4: Build The Full 15-Minute Weather Dataset

Turn raw API output into one clean dataset with UTC timestamps every 15 minutes. This is the full weather table before feature reduction.

In [6]:
# Start from raw hourly weather data and standardize timestamps.
working_df = raw_weather_df.copy()
working_df["time"] = pd.to_datetime(working_df["time"], utc=True, errors="coerce")
working_df = working_df.dropna(subset=["time"]).sort_values("time")
working_df = working_df.drop_duplicates(subset=["time"], keep="first")
working_df = working_df.set_index("time")

weather_cols = [col for col in COMMON_HOURLY_WEATHER_VARS if col in working_df.columns]
for weather_col in weather_cols:
    working_df[weather_col] = pd.to_numeric(working_df[weather_col], errors="coerce")

calendar_index = pd.DatetimeIndex(household_calendar["utc_timestamp"]).tz_convert("UTC")
calendar_index = calendar_index.sort_values().unique()
calendar_index = pd.DatetimeIndex(calendar_index)

combined_index = working_df.index.union(calendar_index)
full_weather_indexed = working_df[weather_cols].reindex(combined_index).sort_index()
full_weather_indexed = full_weather_indexed.interpolate(method="time", limit_direction="both")
full_weather_indexed = full_weather_indexed.ffill().bfill()

full_weather_df = full_weather_indexed.reindex(calendar_index).reset_index()
full_weather_df = full_weather_df.rename(columns={"index": "utc_timestamp"})
full_weather_df["source"] = "open_meteo_archive_interpolated"

print(f"Built full_weather_df with {len(full_weather_df)} rows and {full_weather_df.shape[1]} columns")
print(f"Expected rows from household calendar: {len(calendar_index)}")
print(f"Duplicate timestamps in full_weather_df: {full_weather_df['utc_timestamp'].duplicated().sum()}")

FULL_WEATHER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
full_weather_df.to_csv(FULL_WEATHER_OUTPUT_PATH, index=False)
print(f"Saved full 15-minute weather data to: {FULL_WEATHER_OUTPUT_PATH}")

full_weather_df.head()

Built full_weather_df with 153810 rows and 27 columns
Expected rows from household calendar: 153810
Duplicate timestamps in full_weather_df: 0
Saved full 15-minute weather data to: c:\Users\Mehdi Zamani\hems-automation\data\weather_full_15min.csv


,utc_timestamp,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,...,direct_radiation,diffuse_radiation,global_tilted_irradiance,sunshine_duration,wind_speed_10m,wind_direction_10m,wind_gusts_10m,et0_fao_evapotranspiration,vapour_pressure_deficit,source
0,2014-12-11 17:45:00+00:00,4.875,79.5,1.650,-0.625,0.30,0.30,0.0,0.0,51.0,...,0.0,0.0,0.0,0.0,25.125,237.50,55.925,0.0225,0.1775,open_meteo_archive_interpolated
1,2014-12-11 18:00:00+00:00,4.800,80.0,1.700,-0.700,0.30,0.30,0.0,0.0,51.0,...,0.0,0.0,0.0,0.0,25.100,238.00,58.000,0.0200,0.1700,open_meteo_archive_interpolated
2,2014-12-11 18:15:00+00:00,4.825,80.0,1.725,-0.600,0.35,0.35,0.0,0.0,51.5,...,0.0,0.0,0.0,0.0,24.675,238.25,58.450,0.0200,0.1700,open_meteo_archive_interpolated
3,2014-12-11 18:30:00+00:00,4.850,80.0,1.750,-0.500,0.40,0.40,0.0,0.0,52.0,...,0.0,0.0,0.0,0.0,24.250,238.50,58.900,0.0200,0.1700,open_meteo_archive_interpolated
4,2014-12-11 18:45:00+00:00,4.875,80.0,1.775,-0.400,0.45,0.45,0.0,0.0,52.5,...,0.0,0.0,0.0,0.0,23.825,238.75,59.350,0.0200,0.1700,open_meteo_archive_interpolated


## Step 5: Create Model-Ready Weather Features

Add a small set of useful features that can also be created later from future forecast data.

In [26]:
weather_features_df = full_weather_df.copy()

# Basic derived weather features
weather_features_df["heating_degree_18c"] = np.maximum(18 - weather_features_df["temperature_2m"], 0)
weather_features_df["cooling_degree_22c"] = np.maximum(weather_features_df["temperature_2m"] - 22, 0)
weather_features_df["is_raining"] = (weather_features_df["rain"].fillna(0) > 0).astype(int)
weather_features_df["is_snowing"] = (weather_features_df["snowfall"].fillna(0) > 0).astype(int)
weather_features_df["is_dark"] = (weather_features_df["shortwave_radiation"].fillna(0) < 10).astype(int)

# Wind direction to vector
wind_dir_rad = np.deg2rad(weather_features_df["wind_direction_10m"].fillna(0))
weather_features_df["wind_u_10m"] = weather_features_df["wind_speed_10m"].fillna(0) * np.cos(wind_dir_rad)
weather_features_df["wind_v_10m"] = weather_features_df["wind_speed_10m"].fillna(0) * np.sin(wind_dir_rad)


print("weather_features_df shape:", weather_features_df.shape)
weather_features_df.head()

weather_features_df shape: (153810, 34)


,utc_timestamp,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,...,et0_fao_evapotranspiration,vapour_pressure_deficit,source,heating_degree_18c,cooling_degree_22c,is_raining,is_snowing,is_dark,wind_u_10m,wind_v_10m
0,2014-12-11 17:45:00+00:00,4.875,79.5,1.650,-0.625,0.30,0.30,0.0,0.0,51.0,...,0.0225,0.1775,open_meteo_archive_interpolated,13.125,0.0,1,0,1,-13.499653,-21.190210
1,2014-12-11 18:00:00+00:00,4.800,80.0,1.700,-0.700,0.30,0.30,0.0,0.0,51.0,...,0.0200,0.1700,open_meteo_archive_interpolated,13.200,0.0,1,0,1,-13.300974,-21.286007
2,2014-12-11 18:15:00+00:00,4.825,80.0,1.725,-0.600,0.35,0.35,0.0,0.0,51.5,...,0.0200,0.1700,open_meteo_archive_interpolated,13.175,0.0,1,0,1,-12.984329,-20.982441
3,2014-12-11 18:30:00+00:00,4.850,80.0,1.750,-0.500,0.40,0.40,0.0,0.0,52.0,...,0.0200,0.1700,open_meteo_archive_interpolated,13.150,0.0,1,0,1,-12.670590,-20.676524
4,2014-12-11 18:45:00+00:00,4.875,80.0,1.775,-0.400,0.45,0.45,0.0,0.0,52.5,...,0.0200,0.1700,open_meteo_archive_interpolated,13.125,0.0,1,0,1,-12.359773,-20.368275


## Step 6: Reduce The Feature Set For Residential1

Use historical data to decide which weather features matter most for `residential1_load`. Save the selected columns so the same reduced dataset can be created later.

In [27]:
from lightgbm import LGBMRegressor
import pickle

REDUCTION_TARGET = "residential1_load"

# Merge weather features with target
target_df = household_df[["utc_timestamp", REDUCTION_TARGET]].copy()
merged_df = weather_features_df.merge(target_df, on="utc_timestamp", how="inner")

# Drop rows with missing values
merged_df = merged_df.dropna().reset_index(drop=True)

# Define X and y
drop_cols = ["utc_timestamp", REDUCTION_TARGET, "source"]
drop_cols = [col for col in drop_cols if col in merged_df.columns]

X = merged_df.drop(columns=drop_cols)
y = merged_df[REDUCTION_TARGET]

# Train simple LightGBM model
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
model.fit(X, y)

# Feature importance
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("Top weather features:")
print(importance_df.head(30))

# Keep top features
selected_feature_columns = importance_df.head(30)["feature"].tolist()

print("\nSelected feature columns:")
print(selected_feature_columns)

# Reduced dataset
reduced_weather_df = merged_df[["utc_timestamp", *selected_feature_columns]].copy()

# Save reduced dataset
REDUCED_WEATHER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
reduced_weather_df.to_csv(REDUCED_WEATHER_OUTPUT_PATH, index=False)

# Save feature list/spec
REDUCTION_SPEC_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(REDUCTION_SPEC_PATH, "wb") as f:
    pickle.dump(selected_feature_columns, f)

print(f"\nSaved reduced weather dataset to: {REDUCED_WEATHER_OUTPUT_PATH}")
print(f"Saved feature spec to: {REDUCTION_SPEC_PATH}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018117 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6575
[LightGBM] [Info] Number of data points in the train set: 153810, number of used features: 32
[LightGBM] [Info] Start training from score 0.166557
Top weather features:
                       feature  importance
2                 dew_point_2m         861
22              wind_gusts_10m         774
10            surface_pressure         613
17           diffuse_radiation         529
9                 pressure_msl         516
31                  wind_v_10m         507
1         relative_humidity_2m         444
30                  wind_u_10m         412
20              wind_speed_10m         394
21          wind_direction_10m         348
12             cloud_cover_low         348
3         apparent_temperature         342
13             cloud_cover_mid         320
23  et0_fao_evapotranspiration   

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
import numpy as np

# Time-based split
split_idx = int(len(merged_df) * 0.8)

train_df = merged_df.iloc[:split_idx].copy()
test_df = merged_df.iloc[split_idx:].copy()

# Full feature set
full_drop_cols = ["utc_timestamp", REDUCTION_TARGET, "source"]
full_drop_cols = [col for col in full_drop_cols if col in merged_df.columns]

X_train_full = train_df.drop(columns=full_drop_cols)
X_test_full = test_df.drop(columns=full_drop_cols)
y_train = train_df[REDUCTION_TARGET]
y_test = test_df[REDUCTION_TARGET]

model_full = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
model_full.fit(X_train_full, y_train)
pred_full = model_full.predict(X_test_full)

mae_full = mean_absolute_error(y_test, pred_full)
rmse_full = np.sqrt(mean_squared_error(y_test, pred_full))

# Reduced feature set
X_train_red = train_df[selected_feature_columns]
X_test_red = test_df[selected_feature_columns]

model_red = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
model_red.fit(X_train_red, y_train)
pred_red = model_red.predict(X_test_red)

mae_red = mean_absolute_error(y_test, pred_red)
rmse_red = np.sqrt(mean_squared_error(y_test, pred_red))

print("Full feature set:")
print("MAE:", mae_full)
print("RMSE:", rmse_full)

print("\nReduced feature set:")
print("MAE:", mae_red)
print("RMSE:", rmse_red)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016637 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6538
[LightGBM] [Info] Number of data points in the train set: 123048, number of used features: 32
[LightGBM] [Info] Start training from score 0.208196
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002877 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6534
[LightGBM] [Info] Number of data points in the train set: 123048, number of used features: 30
[LightGBM] [Info] Start training from score 0.208196
Full feature set:
MAE: 0.13988918240447493
RMSE: 0.21250626773649184

Reduced feature set:
MAE: 0.1398550080436823
RMSE: 0.21250407011777026


In [29]:
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMRegressor

param_grid = {
    "num_leaves": [31, 50],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [300, 500, 700],
    "max_depth": [-1, 10],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

model = LGBMRegressor(random_state=42)

search = RandomizedSearchCV(
    model,
    param_grid,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=2,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train_red, y_train)

print("Best params:")
print(search.best_params_)

Fitting 2 folds for each of 15 candidates, totalling 30 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022125 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6534
[LightGBM] [Info] Number of data points in the train set: 123048, number of used features: 30
[LightGBM] [Info] Start training from score 0.208196
Best params:
{'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.03, 'colsample_bytree': 0.7}


In [30]:
best_model = search.best_estimator_

pred = best_model.predict(X_test_red)

from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, pred))

print("Tuned RMSE:", rmse)

Tuned RMSE: 0.21748902023953462


## Step 7: Standardized Inference Pipeline for Future Weather Data

Future forecast data must follow the exact same preprocessing pipeline:

1. Fetch forecast weather data from API
2. Convert to 15-minute UTC time series
3. Apply identical feature engineering steps
4. Select only pre-defined feature columns (selected_feature_columns)
5. Ensure column order and schema consistency with training data

In [31]:
def build_reduced_weather_features_from_full(full_weather_input_df, selected_feature_columns):
    """
    Take a full 15-minute weather dataframe and return the reduced feature dataframe
    with the same structure used during model training.
    """
    weather_features_df = full_weather_input_df.copy()

    # Basic derived weather features
    weather_features_df["heating_degree_18c"] = np.maximum(18 - weather_features_df["temperature_2m"], 0)
    weather_features_df["cooling_degree_22c"] = np.maximum(weather_features_df["temperature_2m"] - 22, 0)
    weather_features_df["is_raining"] = (weather_features_df["rain"].fillna(0) > 0).astype(int)
    weather_features_df["is_snowing"] = (weather_features_df["snowfall"].fillna(0) > 0).astype(int)
    weather_features_df["is_dark"] = (weather_features_df["shortwave_radiation"].fillna(0) < 10).astype(int)

    # Wind direction → vector
    wind_dir_rad = np.deg2rad(weather_features_df["wind_direction_10m"].fillna(0))
    weather_features_df["wind_u_10m"] = weather_features_df["wind_speed_10m"].fillna(0) * np.cos(wind_dir_rad)
    weather_features_df["wind_v_10m"] = weather_features_df["wind_speed_10m"].fillna(0) * np.sin(wind_dir_rad)

    # Time features
    weather_features_df["hour"] = weather_features_df["utc_timestamp"].dt.hour
    weather_features_df["dayofweek"] = weather_features_df["utc_timestamp"].dt.dayofweek
    weather_features_df["is_weekend"] = weather_features_df["dayofweek"].isin([5, 6]).astype(int)

    # Keep only selected features in the saved order
    missing_features = [col for col in selected_feature_columns if col not in weather_features_df.columns]
    if missing_features:
        raise ValueError(f"Missing required selected features: {missing_features}")

    reduced_df = weather_features_df[["utc_timestamp", *selected_feature_columns]].copy()
    return reduced_df

In [32]:
# Simulate future-call reuse by applying the same reduction logic
future_reduced_df = build_reduced_weather_features_from_full(
    full_weather_input_df=full_weather_df,
    selected_feature_columns=selected_feature_columns
)

print("future_reduced_df shape:", future_reduced_df.shape)
future_reduced_df.head()

future_reduced_df shape: (153810, 31)


,utc_timestamp,dew_point_2m,wind_gusts_10m,surface_pressure,diffuse_radiation,pressure_msl,wind_v_10m,relative_humidity_2m,wind_u_10m,wind_speed_10m,...,temperature_2m,weather_code,sunshine_duration,snow_depth,precipitation,cooling_degree_22c,heating_degree_18c,rain,snowfall,is_raining
0,2014-12-11 17:45:00+00:00,1.650,55.925,970.000,0.0,1019.775,-21.190210,79.5,-13.499653,25.125,...,4.875,51.0,0.0,0.0,0.30,0.0,13.125,0.30,0.0,1
1,2014-12-11 18:00:00+00:00,1.700,58.000,970.100,0.0,1019.900,-21.286007,80.0,-13.300974,25.100,...,4.800,51.0,0.0,0.0,0.30,0.0,13.200,0.30,0.0,1
2,2014-12-11 18:15:00+00:00,1.725,58.450,970.225,0.0,1020.025,-20.982441,80.0,-12.984329,24.675,...,4.825,51.5,0.0,0.0,0.35,0.0,13.175,0.35,0.0,1
3,2014-12-11 18:30:00+00:00,1.750,58.900,970.350,0.0,1020.150,-20.676524,80.0,-12.670590,24.250,...,4.850,52.0,0.0,0.0,0.40,0.0,13.150,0.40,0.0,1
4,2014-12-11 18:45:00+00:00,1.775,59.350,970.475,0.0,1020.275,-20.368275,80.0,-12.359773,23.825,...,4.875,52.5,0.0,0.0,0.45,0.0,13.125,0.45,0.0,1


## Final Data Validation and Artifact Export

The pipeline produces the following artifacts:

- raw_weather_dataset: original hourly API data
- full_weather_dataset: cleaned and interpolated 15-minute dataset
- reduced_weather_dataset: final feature set for modeling
- reduction_spec: saved feature selection schema

Before saving, enforce the following checks:

- timestamps are strictly in UTC format
- no duplicate timestamps exist
- no missing values in selected features
- reduced dataset columns exactly match the saved feature specification
- column order is consistent for model input

In [33]:
import pickle

OUTPUT_PATHS = {
    "raw_weather_dataset": RAW_WEATHER_OUTPUT_PATH,
    "full_weather_dataset": FULL_WEATHER_OUTPUT_PATH,
    "reduced_weather_dataset": REDUCED_WEATHER_OUTPUT_PATH,
    "reduction_spec": REDUCTION_SPEC_PATH,
}

# Final checks
assert full_weather_df["utc_timestamp"].dt.tz is not None, "full_weather_df timestamps must be timezone-aware"
assert future_reduced_df["utc_timestamp"].dt.tz is not None, "future_reduced_df timestamps must be timezone-aware"

assert full_weather_df["utc_timestamp"].duplicated().sum() == 0, "Duplicate timestamps in full_weather_df"
assert future_reduced_df["utc_timestamp"].duplicated().sum() == 0, "Duplicate timestamps in future_reduced_df"

expected_columns = ["utc_timestamp", *selected_feature_columns]
assert list(future_reduced_df.columns) == expected_columns, "Reduced future dataset columns do not match selected features"

# Save reduced dataset again to ensure final structure is stored
future_reduced_df.to_csv(REDUCED_WEATHER_OUTPUT_PATH, index=False)

# Save reduction spec
with open(REDUCTION_SPEC_PATH, "wb") as f:
    pickle.dump(selected_feature_columns, f)

print("Final checks passed.")
print("Saved outputs:")
for name, path in OUTPUT_PATHS.items():
    print(f"- {name}: {path}")

Final checks passed.
Saved outputs:
- raw_weather_dataset: c:\Users\Mehdi Zamani\hems-automation\data\weather_raw_hourly.csv
- full_weather_dataset: c:\Users\Mehdi Zamani\hems-automation\data\weather_full_15min.csv
- reduced_weather_dataset: c:\Users\Mehdi Zamani\hems-automation\data\weather_reduced_residential1.csv
- reduction_spec: c:\Users\Mehdi Zamani\hems-automation\models\weather_feature_spec_residential1.pkl


In [34]:
# Step 8: Train Final Load Forecast Model and Save

from lightgbm import LGBMRegressor
import pickle

# Train on full reduced dataset
X_all = merged_df[selected_feature_columns]
y_all = merged_df[REDUCTION_TARGET]

final_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

final_model.fit(X_all, y_all)

# Save model
MODEL_PATH = PROJECT_ROOT / "models" / "load_forecast_model_residential1.pkl"

with open(MODEL_PATH, "wb") as f:
    pickle.dump(final_model, f)

print(f"Final model saved to: {MODEL_PATH}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014529 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6571
[LightGBM] [Info] Number of data points in the train set: 153810, number of used features: 30
[LightGBM] [Info] Start training from score 0.166557
Final model saved to: c:\Users\Mehdi Zamani\hems-automation\models\load_forecast_model_residential1.pkl
